<a href="https://colab.research.google.com/github/furkanaras0/Tez-recipe-recommender-system/blob/main/FoodcomTFRS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==========================================
# 1. KURULUM VE KÜTÜPHANELER
# ==========================================
!pip install -q tensorflow-recommenders tf-keras sentence-transformers

import os
os.environ["TF_USE_LEGACY_KERAS"] = "1" # Keras 2 uyumluluğu

from google.colab import drive
drive.mount('/content/drive')
!unzip -q -o /content/drive/MyDrive/Tez/food-com-recipes-and-user-interactions.zip -d /content/

import numpy as np
import pandas as pd
import ast
import scipy.sparse as sp
import tensorflow as tf
import tensorflow_recommenders as tfrs
from sentence_transformers import SentenceTransformer

from tqdm import tqdm
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import euclidean_distances, cosine_similarity
from sklearn.model_selection import train_test_split

DATA_PATH = "/content"
print("✅ TFRS, SBERT ve gerekli tüm kütüphaneler yüklendi.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ TFRS, SBERT ve gerekli tüm kütüphaneler yüklendi.


In [ ]:
# ==========================================
# 2. VERİ YÜKLEME VE %80 TRAIN - %20 TEST AYRIMI
# ==========================================
print("🔄 Veriler yükleniyor ve temizleniyor...")

interactions = pd.read_csv(DATA_PATH + "/RAW_interactions.csv", usecols=["user_id", "recipe_id", "rating"], dtype={"user_id": "int32", "recipe_id": "int32", "rating": "float32"})
recipes = pd.read_csv(DATA_PATH + "/RAW_recipes.csv", dtype={"id": "int32"}, engine='python', on_bad_lines='skip')

# 1. Temizlik
interactions = interactions[interactions["rating"] > 0]
interactions = interactions.drop_duplicates(subset=["user_id", "recipe_id"], keep="last").dropna()
recipes = recipes.dropna(subset=['id', 'name'])

valid_recipe_ids = set(recipes["id"].unique())
interactions = interactions[interactions["recipe_id"].isin(valid_recipe_ids)].reset_index(drop=True)

# 2. Min 3 Etkileşim Filtresi
user_counts = interactions['user_id'].value_counts()
active_users = user_counts[user_counts >= 3].index
interactions_filtered = interactions[interactions['user_id'].isin(active_users)].copy()

# 3. Train / Test Ayrımı
train_df, test_df = train_test_split(interactions_filtered, test_size=0.20, random_state=42, shuffle=True)

# 4. Cold-Start Koruması
train_users = set(train_df["user_id"].unique())
train_items = set(train_df["recipe_id"].unique())

test_df = test_df[test_df["user_id"].isin(train_users) & test_df["recipe_id"].isin(train_items)].reset_index(drop=True)
train_df = train_df.reset_index(drop=True)
recipes = recipes[recipes["id"].isin(train_items)].reset_index(drop=True)

print(f"✅ Eğitim Seti: {len(train_df)} etkileşim | Test Seti: {len(test_df)} etkileşim")

🔄 Veriler yükleniyor ve temizleniyor...
✅ Eğitim Seti: 712456 etkileşim | Test Seti: 155744 etkileşim


In [ ]:
# ==========================================
# 3. ÖZELLİK ÇIKARIMI (SBERT VE TF-IDF)
# ==========================================
print("🔄 Özellikler işleniyor...")

def safe_literal_eval(x):
    if pd.isna(x) or x == "": return []
    try: return x if isinstance(x, list) else ast.literal_eval(x)
    except: return []

recipes["tags"] = recipes["tags"].apply(safe_literal_eval)
recipes["ingredients"] = recipes["ingredients"].apply(safe_literal_eval)
recipes["nutrition"] = recipes["nutrition"].apply(safe_literal_eval)

recipes["minutes"] = pd.to_numeric(recipes["minutes"], errors="coerce").fillna(0).clip(lower=0)
recipes["time_bucket"] = pd.cut(recipes["minutes"], bins=[-1, 15, 30, 60, float("inf")], labels=["very_fast", "fast", "medium", "long"]).astype(str)

recipes["calories"] = recipes["nutrition"].apply(lambda x: float(x[0]) if isinstance(x, list) and len(x)>0 else 0.0)
recipes["calorie_bucket"] = pd.cut(recipes["calories"], bins=[-1, 200, 500, float("inf")], labels=["low_cal", "medium_cal", "high_cal"]).astype(str)

# Metin Birleştirme
recipes["content_text"] = recipes["name"].fillna("") + " " + recipes["tags"].apply(lambda x: " ".join(x)) + " " + recipes["ingredients"].apply(lambda x: " ".join(x))

# 1. METRİKLER İÇİN: TF-IDF (Sadece hesaplama için, modele GİRMEYECEK)
tfidf = TfidfVectorizer(max_features=10000, stop_words='english')
tfidf_matrix = tfidf.fit_transform(recipes['content_text'])
recipe_id_to_tfidf_idx = {str(rid): i for i, rid in enumerate(recipes['id'])}

# 2. MODEL İÇİN: SBERT (Sentence-BERT) Zekası (Modele GİRECEK)
print("\n🧠 SBERT Semantik Vektörleri Çıkarılıyor (GPU ile 2-4 dk)...")
# Çok hızlı ve son derece başarılı bir model
sbert_model = SentenceTransformer('all-MiniLM-L6-v2')

# 384 Boyutlu Yoğun (Dense) Vektörler
sbert_embeddings = sbert_model.encode(recipes['content_text'].tolist(), show_progress_bar=True)
recipe_id_to_sbert = {str(row['id']): sbert_embeddings[i] for i, row in recipes.iterrows()}

print(f"✅ SBERT vektörleşmesi tamamlandı! (Boyut: {sbert_embeddings.shape})")

🔄 Özellikler işleniyor...

🧠 SBERT Semantik Vektörleri Çıkarılıyor (GPU ile 2-4 dk)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/5994 [00:00<?, ?it/s]

✅ SBERT vektörleşmesi tamamlandı! (Boyut: (191780, 384))


In [ ]:
# ==========================================
# 4. TFRS TENSOR VERİ SETİ (SBERT ENTEGRELİ)
# ==========================================
print("🔄 tf.data.Dataset oluşturuluyor...")

train_df["user_id"] = train_df["user_id"].astype(str)
train_df["recipe_id"] = train_df["recipe_id"].astype(str)
test_df["user_id"] = test_df["user_id"].astype(str)
test_df["recipe_id"] = test_df["recipe_id"].astype(str)
recipes["id"] = recipes["id"].astype(str)

#SBERT tarifin içeriğini (ne olduğunu) anlarken; ID'leri sayı yerine string (metin) yapmak,
#hem TensorFlow'un bu kimlik numaralarını hatalı birer matematiksel büyüklük (örn: 5000 > 150)
#gibi hesaplamasını engeller hem de modelin kullanıcı ve tariflere ait o benzersiz etkileşim geçmişini
#(işbirlikçi filtrelemeyi) kaybetmeden kusursuz bir hibrit sistem kurmasını sağlar.

train_merged = train_df.merge(recipes[['id', 'time_bucket', 'calorie_bucket']], left_on='recipe_id', right_on='id')

# Train için SBERT vektörlerini Numpy dizisi olarak hizala
train_sbert_vectors = np.array([recipe_id_to_sbert[rid] for rid in train_merged["recipe_id"].values])

train_ds = tf.data.Dataset.from_tensor_slices({
    "user_id": train_merged["user_id"].values,
    "recipe_id": train_merged["recipe_id"].values,
    "time_bucket": train_merged["time_bucket"].values,
    "calorie_bucket": train_merged["calorie_bucket"].values,
    "sbert_vector": train_sbert_vectors # SBERT vektörü doğrudan Tensor'a veriliyor
}).shuffle(100000).batch(2048).cache()

# Adaylar (Candidates) için SBERT hizalaması
all_sbert_vectors = np.array([recipe_id_to_sbert[rid] for rid in recipes["id"].values])

recipes_ds = tf.data.Dataset.from_tensor_slices({
    "recipe_id": recipes["id"].values,
    "time_bucket": recipes["time_bucket"].values,
    "calorie_bucket": recipes["calorie_bucket"].values,
    "sbert_vector": all_sbert_vectors
}).batch(2048).cache()

unique_user_ids = train_df["user_id"].unique()
unique_recipe_ids = recipes["id"].unique()
unique_time_buckets = recipes["time_bucket"].unique()
unique_cal_buckets = recipes["calorie_bucket"].unique()

print("✅ Tensor veri setleri hazır.")

🔄 tf.data.Dataset oluşturuluyor...
✅ Tensor veri setleri hazır.


In [ ]:
# ==========================================
# 5. SBERT DESTEKLİ TFRS MODEL MİMARİSİ
# ==========================================
print("🧠 SBERT Destekli TFRS İki Kuleli Model Kuruluyor...")

EMBEDDING_DIM = 128

class UserModel(tf.keras.Model):
    def __init__(self):
        super().__init__()
        self.user_embedding = tf.keras.Sequential([
            tf.keras.layers.StringLookup(vocabulary=unique_user_ids, mask_token=None),
            tf.keras.layers.Embedding(len(unique_user_ids) + 1, EMBEDDING_DIM)
        ])

    def call(self, inputs):
        return tf.math.l2_normalize(self.user_embedding(inputs), axis=1)

class RecipeModel(tf.keras.Model):
    def __init__(self):
        super().__init__()
        self.id_emb = tf.keras.Sequential([
            tf.keras.layers.StringLookup(vocabulary=unique_recipe_ids, mask_token=None),
            tf.keras.layers.Embedding(len(unique_recipe_ids) + 1, 64)
        ])
        self.time_emb = tf.keras.Sequential([
            tf.keras.layers.StringLookup(vocabulary=unique_time_buckets, mask_token=None),
            tf.keras.layers.Embedding(len(unique_time_buckets) + 1, 16)
        ])
        self.cal_emb = tf.keras.Sequential([
            tf.keras.layers.StringLookup(vocabulary=unique_cal_buckets, mask_token=None),
            tf.keras.layers.Embedding(len(unique_cal_buckets) + 1, 16)
        ])

        # MUCİZE BURADA: 384 boyutlu hazır SBERT vektörünü modelin anlayacağı dile çeviriyoruz.
        # Dropout, aşırı ezberlemeyi engeller.
        self.sbert_dense = tf.keras.Sequential([
            tf.keras.layers.Dense(128, activation='relu'),
            tf.keras.layers.Dropout(0.2),
            tf.keras.layers.Dense(64, activation='relu')
        ])

        self.final_dense = tf.keras.layers.Dense(EMBEDDING_DIM)

    def call(self, inputs):
        sbert_features = self.sbert_dense(inputs["sbert_vector"])

        x = self.final_dense(tf.concat([
            self.id_emb(inputs["recipe_id"]),
            self.time_emb(inputs["time_bucket"]),
            self.cal_emb(inputs["calorie_bucket"]),
            sbert_features
        ], axis=1))
        return tf.math.l2_normalize(x, axis=1)

class FoodRecommender(tfrs.Model):
    def __init__(self):
        super().__init__()
        self.user_model = UserModel()
        self.recipe_model = RecipeModel()
        self.task = tfrs.tasks.Retrieval(
            metrics=tfrs.metrics.FactorizedTopK(
                candidates=recipes_ds.map(self.recipe_model)
            )
        )

    def compute_loss(self, features, training=False):
        user_embeddings = self.user_model(features["user_id"])
        recipe_embeddings = self.recipe_model({
            "recipe_id": features["recipe_id"],
            "time_bucket": features["time_bucket"],
            "calorie_bucket": features["calorie_bucket"],
            "sbert_vector": features["sbert_vector"]
        })
        return self.task(user_embeddings, recipe_embeddings)

print("✅ Model mimarisi tanımlandı.")

🧠 SBERT Destekli TFRS İki Kuleli Model Kuruluyor...
✅ Model mimarisi tanımlandı.


In [ ]:
# ==========================================
# 6. EĞİTİM, İNDEKSLEME VE "ISINMALI" KAYIT
# ==========================================
model = FoodRecommender()
model.compile(optimizer=tf.keras.optimizers.Adagrad(learning_rate=0.05))

EPOCH_SAYISI = 50
erken_durdurma = tf.keras.callbacks.EarlyStopping(monitor='loss', patience=3, restore_best_weights=True)

print(f"\n🚀 SBERT Destekli TFRS Eğitimi Başlıyor...\n")
model.fit(train_ds, epochs=EPOCH_SAYISI, callbacks=[erken_durdurma])

print("\n💾 Tahmin İndeksi Kuruluyor (BruteForce)...")
index = tfrs.layers.factorized_top_k.BruteForce(model.user_model, k=100)
index.index_from_dataset(
    tf.data.Dataset.zip((recipes_ds.map(lambda x: x["recipe_id"]), recipes_ds.map(model.recipe_model)))
)

# 🛡️ İŞTE HAYAT KURTARAN SİHİRLİ DOKUNUŞ (WARM-UP)
# Modeli kaydetmeden hemen önce sahte bir kullanıcıyla tahmin yaptırıyoruz.
# Böylece TensorFlow tahmin haritasını (Graph) çizip hafızaya alıyor!
print("🔥 Model kaydetmeden önce ısınma turuna çıkarılıyor...")
_ = index(tf.constant(["dummy_user_123"]))

# === KAYIT İŞLEMLERİ ===
import pickle
import os
import scipy.sparse as sp

SAVE_DIR = '/content/drive/MyDrive/Tez3/SavedTFRS/'
os.makedirs(SAVE_DIR, exist_ok=True)

tf.saved_model.save(index, SAVE_DIR + "tfrs_bruteforce_index")
sp.save_npz(SAVE_DIR + 'tfidf_matrix.npz', tfidf_matrix)

with open(SAVE_DIR + 'recipe_id_to_tfidf_idx.pkl', 'wb') as f:
    pickle.dump(recipe_id_to_tfidf_idx, f)

recipes.to_pickle(SAVE_DIR + 'recipes_processed.pkl')
train_df.to_pickle(SAVE_DIR + 'train_df.pkl')
test_df.to_pickle(SAVE_DIR + 'test_df.pkl')

print(f"✅ SBERT-TFRS Modeli ve Veriler Başarıyla {SAVE_DIR} Konumuna Kaydedildi!")


🚀 SBERT Destekli TFRS Eğitimi Başlıyor...

Epoch 1/50
348/348 [==============================] - 544s 2s/step - factorized_top_k/top_1_categorical_accuracy: 0.0011 - factorized_top_k/top_5_categorical_accuracy: 0.0018 - factorized_top_k/top_10_categorical_accuracy: 0.0023 - factorized_top_k/top_50_categorical_accuracy: 0.0050 - factorized_top_k/top_100_categorical_accuracy: 0.0075 - loss: 15253.9813 - regularization_loss: 0.0000e+00 - total_loss: 15253.9813
Epoch 2/50
348/348 [==============================] - 536s 2s/step - factorized_top_k/top_1_categorical_accuracy: 1.8949e-04 - factorized_top_k/top_5_categorical_accuracy: 0.0010 - factorized_top_k/top_10_categorical_accuracy: 0.0018 - factorized_top_k/top_50_categorical_accuracy: 0.0066 - factorized_top_k/top_100_categorical_accuracy: 0.0111 - loss: 14895.7582 - regularization_loss: 0.0000e+00 - total_loss: 14895.7582
Epoch 3/50
348/348 [==============================] - 535s 2s/step - factorized_top_k/top_1_categorical_accuracy: 

✅ SBERT-TFRS Modeli ve Veriler Başarıyla /content/drive/MyDrive/Tez3/SavedTFRS/ Konumuna Kaydedildi!


In [ ]:
# ==========================================
# 7. METRİK DEĞERLENDİRME (GLOBAL AUC İLE)
# ==========================================
import numpy as np
from tqdm import tqdm
from sklearn.metrics import roc_auc_score
from sklearn.metrics.pairwise import euclidean_distances, cosine_similarity
import tensorflow as tf

print("\n⚙️ 10.000 Kullanıcı İçin Kapsamlı Metrikler (Global AUC dahil) Hesaplanıyor...")

train_user_items = train_df.groupby("user_id")["recipe_id"].apply(set).to_dict()
test_user_items = test_df.groupby("user_id")["recipe_id"].apply(set).to_dict()

def tfrs_evaluate_global_metrics(index_model, test_users, train_history, test_history, tfidf_matrix, rec_to_tfidf, k=10, batch_size=256):
    precisions, recalls, aucs, hd_scores, diversities = [], [], [], [], []
    recommended_items = set()
    total_items = len(recipes)

    users_to_test = list(test_users.keys())
    if len(users_to_test) > 10000:
        np.random.seed(42)
        users_to_test = np.random.choice(users_to_test, 10000, replace=False)

    for start in tqdm(range(0, len(users_to_test), batch_size), desc="TFRS Metrikleri"):
        batch_users = users_to_test[start:start+batch_size]

        # GLOBAL AUC İÇİN DÜZELTME: Tüm tarifleri (total_items) çekiyoruz ki LightFM ile adil kıyaslansın!
        scores_batch, predicted_ids_batch = index_model(tf.constant(batch_users), k=total_items)

        for i, user_id in enumerate(batch_users):
            true_items = test_history.get(user_id, set())
            if not true_items: continue

            known_items = train_history.get(user_id, set())

            user_preds_raw = [pid.decode('utf-8') for pid in predicted_ids_batch[i].numpy()]
            user_scores_raw = scores_batch[i].numpy()

            valid_preds = []
            valid_scores = []
            # Train setindeki (zaten yediği) tarifleri gizle
            for pid, score in zip(user_preds_raw, user_scores_raw):
                if pid not in known_items:
                    valid_preds.append(pid)
                    valid_scores.append(score)

            # --- 1. GLOBAL AUC ---
            # Artık havuzda sadece 100 aday değil, tüm tarifler var
            y_true_cand = [1 if pid in true_items else 0 for pid in valid_preds]
            if len(set(y_true_cand)) > 1:
                try:
                    auc = roc_auc_score(y_true_cand, valid_scores)
                    aucs.append(auc)
                except ValueError:
                    pass

            # --- 2. PRECISION, RECALL VE COVERAGE İÇİN İLK K (10) ADETİ SEÇ ---
            top_k_preds = valid_preds[:k]
            recommended_items.update(top_k_preds)

            hits = len(set(top_k_preds).intersection(true_items))
            precisions.append(hits / k)
            recalls.append(hits / len(true_items))

            # --- 3. HD95 VE DIVERSITY (TF-IDF CETVELİ İLE) ---
            pred_tfidf_indices = [rec_to_tfidf[rid] for rid in top_k_preds if rid in rec_to_tfidf]
            true_tfidf_indices = [rec_to_tfidf[rid] for rid in true_items if rid in rec_to_tfidf]

            if len(pred_tfidf_indices) > 0 and len(true_tfidf_indices) > 0:
                pred_vecs = tfidf_matrix[pred_tfidf_indices]
                true_vecs = tfidf_matrix[true_tfidf_indices]

                # HD95
                d_p2t = np.min(euclidean_distances(pred_vecs, true_vecs), axis=1)
                d_t2p = np.min(euclidean_distances(true_vecs, pred_vecs), axis=1)
                hd95_val = max(np.percentile(d_p2t, 95), np.percentile(d_t2p, 95))
                hd_scores.append(hd95_val)

                # Diversity
                if len(pred_tfidf_indices) >= 2:
                    sim_matrix = cosine_similarity(pred_vecs)
                    upper = sim_matrix[np.triu_indices(len(pred_tfidf_indices), k=1)]
                    diversities.append(1 - np.mean(upper))

    mean_p = np.mean(precisions)
    mean_r = np.mean(recalls)
    return mean_p, mean_r, np.mean(aucs), np.mean(hd_scores), np.mean(diversities), len(recommended_items) / total_items

# Çıktıları Global AUC ile paketliyoruz
p, r, auc_global, hd95, div, cov = tfrs_evaluate_global_metrics(
    index_model=index, test_users=test_user_items, train_history=train_user_items,
    test_history=test_user_items, tfidf_matrix=tfidf_matrix, rec_to_tfidf=recipe_id_to_tfidf_idx, k=10
)

print("\n" + "="*45)
print(f"🎯 SBERT-TFRS TEZ SONUÇLARI (ADİL KIYASLAMA)")
print("="*45)
print(f"Precision@10 : {p:.4f}")
print(f"Recall@10    : {r:.4f}")
print(f"Global AUC   : {auc_global:.4f}")
print(f"HD95 (TF-IDF): {hd95:.4f}")
print(f"Diversity@10 : {div:.4f}")
print(f"Coverage     : {cov:.4f}")
print("="*45)


⚙️ 10.000 Kullanıcı İçin Kapsamlı Metrikler (Global AUC dahil) Hesaplanıyor...


TFRS Metrikleri: 100%|██████████| 40/40 [24:14<00:00, 36.37s/it]


🎯 SBERT-TFRS TEZ SONUÇLARI (ADİL KIYASLAMA)
Precision@10 : 0.0010
Recall@10    : 0.0028
Global AUC   : 0.7308
HD95 (TF-IDF): 1.3782
Diversity@10 : 0.9081
Coverage     : 0.3067


In [ ]:
# =============================================
# 🎯 SBERT-TFRS TEZ SONUÇLARI (ADİL KIYASLAMA)
# =============================================
# Precision@10 : 0.0010
# Recall@10    : 0.0028
# Global AUC   : 0.7308
# HD95 (TF-IDF): 1.3782
# Diversity@10 : 0.9081
# Coverage     : 0.3067
# =============================================